In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

# 1. 定義 Feature Engineering 函數 (保持代碼模組化，符合軟件工程規範)
def engineer_features(df):
    df = df.copy()
    
    # 移除冇預測能力嘅 ID 同名字欄位
    cols_to_drop = ['id', 'CustomerId', 'Surname']
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
    
    # 新增黃金特徵
    df['Age_minus_Tenure'] = df['Age'] - df['Tenure']
    df['Balance_per_Product'] = df['Balance'] / df['NumOfProducts']
    df['Balance_to_Salary'] = df['Balance'] / (df['EstimatedSalary'] + 1)
    df['IsZeroBalance'] = (df['Balance'] == 0).astype(int)
    
    # 新增你提議嘅 方案一：類別組合標籤 (保留原始欄位，新增 Group 標籤)
    df['Active_Product_Group'] = df['IsActiveMember'].astype(str) + '_' + df['NumOfProducts'].astype(str)
    
    return df

# 2. 載入與處理資料
df_raw = pd.read_csv('train.csv')

# 套用特徵工程
df_processed = engineer_features(df_raw)

# 3. 切分資料 (80% Training, 20% Testing)
X = df_processed.drop(columns=['Exited'])
y = df_processed['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. 定義 ColumnTransformer (資料前處理)
categorical_features = ['Geography', 'Gender', 'Active_Product_Group']
numerical_features = [col for col in X.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

# 5. 建立 Stacking 模型
# 底層 (Level 0): 兩款不同運作原理嘅強大樹狀模型
base_estimators = [
    # Random Forest (基於 Bagging，穩定性高)
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)),
    # HistGradientBoosting (Scikit-learn 內建嘅 LightGBM 替代品，基於 Boosting，捕捉細節極強)
    ('hgb', HistGradientBoostingClassifier(max_iter=100, random_state=42))
]

# 頂層 (Level 1): Logistic Regression 作為 Meta-learner
meta_model = LogisticRegression()

stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_model,
    cv=5, # 內部使用 5-Fold Cross Validation 防止 Data Leakage
    n_jobs=-1
)

# 6. 組裝成完整 Pipeline 並開始訓練
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', stacking_clf)
])

print("開始訓練 Stacking Pipeline...")
pipeline.fit(X_train, y_train)

# # 7. 預測與驗證
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

# 輸出驗證結果 (預設門檻 0.5)
print("\n--- 驗證結果 (Hold-out Test Set | 預設門檻 0.5) ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}\n")
print("Classification Report (Threshold = 0.5):")
print(classification_report(y_test, y_pred))

# ----------------- 新增以下部分 -----------------
# 8. 商業優化：調整決策門檻 (Threshold Tuning)
custom_threshold = 0.35
y_pred_tuned = (y_pred_proba >= custom_threshold).astype(int)

print(f"\n--- 調整門檻為 {custom_threshold} 後的表現 (重視 Recall 挽留) ---")
print(classification_report(y_test, y_pred_tuned))



開始訓練 Stacking Pipeline...

--- 驗證結果 (Hold-out Test Set) ---
ROC-AUC Score: 0.8889

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.95      0.92     26023
           1       0.75      0.55      0.63      6984

    accuracy                           0.86     33007
   macro avg       0.82      0.75      0.77     33007
weighted avg       0.86      0.86      0.86     33007



In [6]:
# 將門檻由預設的 0.5 調低至 0.35
custom_threshold = 0.35
y_pred_tuned = (y_pred_proba >= custom_threshold).astype(int)

print(f"--- 調整門檻為 {custom_threshold} 後的表現 ---")
print(classification_report(y_test, y_pred_tuned))


--- 調整門檻為 0.35 後的表現 ---
              precision    recall  f1-score   support

           0       0.90      0.93      0.91     26023
           1       0.69      0.62      0.66      6984

    accuracy                           0.86     33007
   macro avg       0.80      0.77      0.79     33007
weighted avg       0.86      0.86      0.86     33007

